# Full-private-encoder dataset MoE — paired end-to-end ablation

Run this notebook top to bottom to test whether the shared encoder is the limiting factor in the finalized four-dataset MoE. The diagnostic motivating this run found `hard_oracle_dataset - moe_oracle_dataset = +30.507` macro-F1 points, while learned top-1 and dense MoE differed by only `0.598` macro-F1 points. That makes the classification/representation path—not blending—the highest-value intervention.

This run changes the expert representation topology only: each expert receives a complete private `47→128→64` encoder followed by the existing `64→45→classes` head. A fifth `47→128→64` encoder supplies the dataset-blind soft gate. Every private encoder and the gate encoder starts from the exact Stage-A checkpoint used by the existing shared-encoder run. Stage C remains dense, task-driven, and unsupervised by dataset ID.

The primary comparison is therefore `moe_dataset_private_encoders` versus the existing `moe_dataset_soft` checkpoint on the same signed splits and seed. This is an architectural ablation, not a capacity-matched claim: the private model intentionally stores and executes more parameters, and the notebook reports that increase explicitly.


In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = "selimsidan"
GITHUB_REPO = "dataset_moe_nids"
GITHUB_BRANCH = "main"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"

DRIVE_DATA_DIR = "/content/drive/MyDrive/NIDS_datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs"
EXECUTION_MODE = "out_of_core_full"  # out_of_core_full | in_memory_smoke

ACTIVE_DATASETS = [
    "NF-UNSW-NB15-v3",
    "NF-ToN-IoT-v3",
    "NF-BoT-IoT-v3",
    "NF-CICIDS2018-v3",
]

ARCHITECTURE = "moe_dataset_private_encoders"
RUN_NAME = "nfv3_4way_moe_private_encoders_seed0_v1"
SHARED_REFERENCE_RUN = "nfv3_4way_moe_soft_hard_compute_matched_seed0_v1"
SEED = 0

LATENT_DIM = 64
ENCODER_HIDDEN_DIMS = [128]
EXPERT_HIDDEN_DIMS = [45]
DROPOUT = 0.2
GATE_SUPERVISION = "none"
EXPERT_UPDATE_POLICY = "all"  # matches the original run; do not change for this ablation
LAMBDA_BALANCE = 0.1
LAMBDA_EXPERT_ANCHOR = 0.0

EPOCHS_A = 30  # used only for smoke mode; full mode reuses the exact reference Stage-A checkpoint
EPOCHS_B = 30
EPOCHS_C = 30
BATCH_SIZE = 512
FORCE_RESTART = False
RUN_TESTS = True
# ==================================================================


## 1. Secure checkout and environment setup

The GitHub token is passed through a temporary HTTP header and is not persisted in the clone URL or Git configuration. Select a GPU runtime before continuing.


In [ ]:
import base64, os, subprocess, sys, time
from pathlib import Path
try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("This notebook is intended for Google Colab.") from exc
drive.mount("/content/drive")
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f"Add {GITHUB_SECRET_NAME} in Colab Secrets and grant notebook access.")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ | {
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
}
repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
repo_dir = Path("/content") / GITHUB_REPO
if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", repo_url, str(repo_dir)], env=git_env, check=True)
git_env.clear()
token = auth = None
os.chdir(repo_dir)
os.environ["NIDS_DRIVE_BASE"] = DRIVE_DATA_DIR
os.environ["NIDS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
os.environ["NIDS_SCRATCH_DIR"] = "/content/dataset_moe_nids_scratch"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Repository:", repo_dir)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Validate the paired scientific contract

Production mode requires the same four schema-compatible NF-v3 datasets and the completed Stage-A checkpoint from the shared-encoder reference run. Reusing that exact checkpoint prevents pooled pretraining noise from being mistaken for a private-encoder effect. Smoke mode trains a one-epoch local Stage A because its capped data contract is intentionally different.


In [ ]:
import torch
from data.registry import get_spec
from training.config import load_config

if EXECUTION_MODE not in {"out_of_core_full", "in_memory_smoke"}:
    raise ValueError("Unknown EXECUTION_MODE")
if ARCHITECTURE != "moe_dataset_private_encoders":
    raise ValueError("This notebook is specifically for the full-private-encoder MoE")
if len(ACTIVE_DATASETS) != 4 or len(set(ACTIVE_DATASETS)) != 4:
    raise ValueError("The paired experiment requires the exact four-dataset combination")
if EXPERT_UPDATE_POLICY != "all" or GATE_SUPERVISION != "none":
    raise ValueError("Keep Stage-C routing/training equal to the shared reference for this ablation")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU.")
missing = {}
for name in ACTIVE_DATASETS:
    spec = get_spec(name)
    if not any(Path(path).is_file() for path in spec.paths):
        missing[name] = spec.paths
if missing:
    raise FileNotFoundError("Missing datasets:\n" + "\n".join(f"  {name}: {paths}" for name, paths in missing.items()))
aliases = [get_spec(name).feature_alias for name in ACTIVE_DATASETS]
if any(get_spec(name).kind != "file" for name in ACTIVE_DATASETS) or any(value != aliases[0] for value in aliases[1:]):
    raise ValueError("The paired full-data run requires schema-compatible single-file NF-v3 datasets")
if len(aliases[0]) != 47:
    raise ValueError(f"Expected 47 features, found {len(aliases[0])}")
if ENCODER_HIDDEN_DIMS != [128] or LATENT_DIM != 64 or EXPERT_HIDDEN_DIMS != [45]:
    raise ValueError("Keep the reference encoder/head widths unchanged")

preview = load_config("config/default.yaml", ["data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]"])
selected_classes = {"Benign"}
for name in ACTIVE_DATASETS:
    selected_classes.update(value for value in preview["data"]["label_mapping"][name].values() if value is not None)
num_classes = len(selected_classes)
if num_classes != 22:
    raise ValueError(f"Expected the paired 22-class vocabulary, found {num_classes}")

source_stage_a = Path(DRIVE_OUTPUT_DIR) / "checkpoints" / SHARED_REFERENCE_RUN / "stage_a_encoder.pt"
source_results = Path(DRIVE_OUTPUT_DIR) / "results" / SHARED_REFERENCE_RUN
if EXECUTION_MODE == "out_of_core_full" and not source_stage_a.is_file():
    raise FileNotFoundError(f"Reference Stage-A checkpoint not found: {source_stage_a}. Run notebook 00 first.")

print("GPU:", torch.cuda.get_device_name(0))
print("Datasets:", ACTIVE_DATASETS)
print("Classes:", num_classes)
print("Reference Stage A:", source_stage_a if EXECUTION_MODE == "out_of_core_full" else "smoke-local")
print("Run:", RUN_NAME)


In [ ]:
# Exact static budget for the full-data 47-feature / 22-class contract.
datasets = len(ACTIVE_DATASETS)
encoder_params = (47 + 1) * 128 + (128 + 1) * 64
encoder_macs = 47 * 128 + 128 * 64
gate_params = (64 + 1) * datasets
gate_macs = 64 * datasets
head_params = (64 + 1) * 45 + (45 + 1) * num_classes
head_macs = 64 * 45 + 45 * num_classes
private_total_params = encoder_params + gate_params + datasets * (encoder_params + head_params)
private_dense_macs = encoder_macs + gate_macs + datasets * (encoder_macs + head_macs)
shared_total_params = encoder_params + gate_params + datasets * head_params
shared_dense_macs = encoder_macs + gate_macs + datasets * head_macs
assert private_total_params == 88008
assert private_dense_macs == 86776
assert shared_total_params == 30408
assert shared_dense_macs == 29944
print(f"Private dense model: {private_total_params:,} parameters; {private_dense_macs:,} MACs/sample")
print(f"Shared dense reference: {shared_total_params:,} parameters; {shared_dense_macs:,} MACs/sample")
print(f"Increase: {private_total_params/shared_total_params:.2f}× parameters; {private_dense_macs/shared_dense_macs:.2f}× MACs")
print("This notebook tests representation-path benefit; it does not claim capacity matching.")


In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
else:
    print("Tests skipped by configuration.")


## 3. Train private encoders, soft gate, and evaluate

In full-data mode, Stage A is intentionally not rerun. Its exact encoder is loaded into five independent copies: one gate encoder and four expert encoders. Stage B updates one complete private encoder+head at a time on its assigned dataset. Stage C uses the original experiment's dense routing, task-only gate, `expert_update_policy=all`, zero anchoring, all-encoder fine-tuning, fixed epochs, optimizer, and class-weighted objective. Thus the expert encoder topology is the targeted change.


In [ ]:
effective_run_name = RUN_NAME if EXECUTION_MODE == "out_of_core_full" else RUN_NAME + "_smoke"
effective_epochs_a = 1 if EXECUTION_MODE == "in_memory_smoke" else EPOCHS_A
effective_epochs_b = 1 if EXECUTION_MODE == "in_memory_smoke" else EPOCHS_B
effective_epochs_c = 1 if EXECUTION_MODE == "in_memory_smoke" else EPOCHS_C
effective_force_restart = True if EXECUTION_MODE == "in_memory_smoke" else FORCE_RESTART
stages = "[A,B,C]" if EXECUTION_MODE == "in_memory_smoke" else "[B,C]"
overrides = [
    f"run_name={effective_run_name}",
    f"seed={SEED}",
    f"data.split_seed={SEED}",
    f"architecture={ARCHITECTURE}",
    "data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]",
    f"training.stages={stages}",
    "training.device=cuda",
    f"training.force_restart={str(effective_force_restart).lower()}",
    f"training.epochs_a={effective_epochs_a}",
    f"training.epochs_b={effective_epochs_b}",
    f"training.epochs_c={effective_epochs_c}",
    f"training.batch_size={BATCH_SIZE}",
    "training.selection_mode=fixed_epochs",
    f"model.latent_dim={LATENT_DIM}",
    "model.encoder.hidden_dims=[" + ",".join(map(str, ENCODER_HIDDEN_DIMS)) + "]",
    "model.expert.hidden_dims=[" + ",".join(map(str, EXPERT_HIDDEN_DIMS)) + "]",
    f"model.encoder.dropout={DROPOUT}",
    f"model.expert.dropout={DROPOUT}",
    "model.gate.routing=dense",
    f"training.stage_c.gate_supervision={GATE_SUPERVISION}",
    f"training.stage_c.expert_update_policy={EXPERT_UPDATE_POLICY}",
    f"training.stage_c.lambda_expert_anchor={LAMBDA_EXPERT_ANCHOR}",
    "training.stage_c_unfreeze=all",
    f"load_balance.lambda_balance={LAMBDA_BALANCE}",
]
if EXECUTION_MODE == "out_of_core_full":
    overrides.append(f"training.stage_a_checkpoint={source_stage_a}")
module = "training.ooc_run" if EXECUTION_MODE == "out_of_core_full" else "training.run"
cmd = [sys.executable, "-u", "-m", module, "--config", "config/default.yaml"]
if EXECUTION_MODE == "in_memory_smoke":
    cmd += ["--mode", "smoke"]
for override in overrides:
    cmd += ["--set", str(override)]
print("Launching:", " ".join(cmd))
started = time.monotonic()
subprocess.run(cmd, check=True, env=os.environ.copy())
print(f"Training and evaluation completed in {(time.monotonic()-started)/3600:.2f} hours.")


## 4. Paired comparison with the shared-encoder run

The first table reports absolute metrics and private-minus-shared deltas on identical origin rows. The experiment supports the private-encoder hypothesis only if macro-F1 improves materially—not merely overall accuracy—and the improvement is not confined to one majority dataset. Resource costs remain part of the conclusion.


In [ ]:
import pandas as pd
from IPython.display import display

private_result_dir = Path(DRIVE_OUTPUT_DIR) / "results" / effective_run_name
private_overall = pd.read_csv(private_result_dir / "Overall_Metrics.csv")
private_resources = pd.read_csv(private_result_dir / "Resource_Accounting.csv")
print("=== Private-encoder absolute results ===")
display(private_overall)

reference_overall_path = source_results / "Overall_Metrics.csv"
if EXECUTION_MODE == "out_of_core_full" and reference_overall_path.is_file():
    shared_overall = pd.read_csv(reference_overall_path)
    metric_columns = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]
    left = private_overall[["origin", *metric_columns]].set_index("origin").add_prefix("private__")
    right = shared_overall[["origin", *metric_columns]].set_index("origin").add_prefix("shared__")
    comparison = left.join(right, how="inner")
    for metric in metric_columns:
        comparison[f"delta__{metric}"] = comparison[f"private__{metric}"] - comparison[f"shared__{metric}"]
    print("=== Paired private minus shared results ===")
    display(comparison.reset_index())
else:
    comparison = None
    print("Paired result deltas are shown only in full-data mode; reference path:", reference_overall_path)

print("=== Private resource accounting ===")
display(private_resources)
reference_resource_path = source_results / "Resource_Accounting.csv"
if reference_resource_path.is_file():
    shared_resources = pd.read_csv(reference_resource_path)
    columns = ["architecture", "total_parameters", "active_parameters_per_sample_mean", "forward_macs_per_sample_mean", "forward_flops_per_sample_mean"]
    print("=== Resource comparison ===")
    display(pd.concat([shared_resources[columns], private_resources[columns]], ignore_index=True))


In [ ]:
for filename in [
    "Per_Dataset_Metrics.csv", "Per_Class_Metrics.csv",
    "Gate_By_Dataset.csv", "Expert_Performance_By_Dataset.csv",
    "Expert_Utilization.csv", "Confusion_Matrix.csv",
]:
    path = private_result_dir / filename
    if path.is_file():
        print(f"=== {filename} ===")
        frame = pd.read_csv(path)
        if filename == "Per_Class_Metrics.csv" and "is_native_class" in frame.columns:
            frame = frame[frame["is_native_class"]].sort_values(["origin", "f1", "support"])
        display(frame)
print("Detailed artifacts:", private_result_dir)
print("Checkpoints:", Path(DRIVE_OUTPUT_DIR) / "checkpoints" / effective_run_name)


## Decision rule and follow-up

Treat this as evidence for private encoders when the combined and per-dataset macro-F1 gains are material and repeat across paired seeds. A gain accompanied only by the roughly `2.89×` parameter and `2.90×` MAC increase is promising but not yet a fair-efficiency victory; the next experiment should narrow the private encoders or compare against a parameter-matched dense model.

This notebook intentionally keeps `expert_update_policy=all` because the original shared-encoder run used it. If private encoders improve performance, a second ablation may switch only that setting to `assigned_only` to test whether preserving Stage-B ownership adds further benefit. Likewise, SupCon/center/triplet objectives from the accompanying discussion should be evaluated later as separate Stage-A interventions, not mixed into this topology test.

For additional seeds, change both `SEED` and `RUN_NAME`, and point `SHARED_REFERENCE_RUN` to the corresponding same-seed shared checkpoint. Keep `FORCE_RESTART=False` after interruptions; use `True` only to deliberately discard this run's compatible stage progress.
